In [18]:
#Cell 1 — import
from src.protogen_preprocessing import (
    load_protogen_data,
    filter_annotation_ra_bl,
    select_relevant_measurement_columns,
    add_marker_name,
    transpose_measurement,
    merge_with_annotation,
    split_metadata_features,
    inspect_missing_values,
    inspect_low_variance_features,
    inspect_high_correlation_features,
    remove_high_correlation_features,
)
import pandas as pd
import numpy as np

In [19]:
#Cell 2 — load
data = load_protogen_data()

df_measurement = data["df_measurement"]
df_annotation = data["df_annotation"]

Protogen sheets loaded.
Measurement shape: (163, 597)
Annotation shape: (593, 63)


In [20]:
#Cell 3 — inspect annotation quickly
print(df_annotation.columns.tolist())
df_annotation.head()

['SampleId', 'Timepoint', 'Patient_ID', 'Digest', 'Study', 'IM.STEROIDS.3MONTHS', 'ACPA.POSITIVE', 'RHUEMATOID.FACTOR', 'DAS28.0M', 'DAS28.3M', 'DAS28.6M', 'DAS28.9M', 'DAS28.12M', 'DAS28.18M', 'HAQ.0M', 'HAQ.6M', 'SDAI.0M', 'SDAI.6M', 'SDAI.12M', 'BASOPHILS.0M', 'EOSINOPHILS.0M', 'HB.0M', 'LYMPHOCYTES.0M', 'MONOCYTES.0M', 'NEUTROPHILS.0M', 'PLT.0M', 'WBC.0M', 'CRP.0M', 'ESR.0M', 'FATIQUE.0M', 'PAIN.0M', 'TOTAL.SWOLLEN.0M', 'TOTAL.TENDER.0M', 'BASOPHILS.6M', 'EOSINOPHILS.6M', 'HB.6M', 'LYMPHOCYTES.6M', 'MONOCYTES.6M', 'NEUTROPHILS.6M', 'PLT.6M', 'WBC.6M', 'CRP.6M', 'FATIQUE.6M', 'PAIN.6M', 'TOTAL.SWOLLEN.6M', 'TOTAL.TENDER.6M', 'CRP.9M', 'TOTAL.SWOLLEN.9M', 'TOTAL.TENDER.9M', 'ORAL.STEROIDS.3M', 'AGE', 'RACE', 'GENDER', 'HEIGHT', 'WEIGHT', 'ALCOHOL_Y_N', 'CURENT SMOKER', 'Symp_Duration', 'Initial.Score', 'Final.Score', 'Erosive', 'Hep B serology wk 9 (IU/mL)', 'vaccine centre']


,SampleId,Timepoint,Patient_ID,Digest,Study,IM.STEROIDS.3MONTHS,ACPA.POSITIVE,RHUEMATOID.FACTOR,DAS28.0M,DAS28.3M,...,HEIGHT,WEIGHT,ALCOHOL_Y_N,CURENT SMOKER,Symp_Duration,Initial.Score,Final.Score,Erosive,Hep B serology wk 9 (IU/mL),vaccine centre
0,TAC1241_BL,BL,TAC1241,608CAC12FA322B4798E2E772CE6C9F0087B4933ADB8C3A...,TACERA,yes,No ...,Yes ...,6.91,7.02,...,NaN,56.6,No ...,Yes,137,9,18,0,NaN,NaN
1,TAC1241_M6,M6,TAC1241,608CAC12FA322B4798E2E772CE6C9F0087B4933ADB8C3A...,TACERA,yes,No ...,Yes ...,6.91,7.02,...,NaN,56.6,No ...,Yes,137,9,18,0,NaN,NaN
2,TAC1147_BL,BL,TAC1147,E856EB9AC7C85B70C9F42C4F6786775FE65DCDFDE2FB71...,TACERA,ND,ND,Yes ...,7.45,3.09,...,175,121,Yes ...,No,84,36,39,1,NaN,NaN
3,TAC1147_M6,M6,TAC1147,E856EB9AC7C85B70C9F42C4F6786775FE65DCDFDE2FB71...,TACERA,ND,ND,Yes ...,7.45,3.09,...,175,121,Yes ...,No,84,36,39,1,NaN,NaN
4,TAC1094_BL,BL,TAC1094,10433449E77D5E82E36A52EC5518583272BB0FD55D9BC4...,TACERA,ND,Yes ...,Yes ...,5.1,ND,...,190,70.1,Yes ...,Yes,155,0,NaN,0,NaN,NaN


In [21]:
#Cell 4 — filter to TACERA + BL + M6
df_annotation_ra_bl = filter_annotation_ra_bl(df_annotation)
df_annotation_ra_bl.head()

Filtered annotation to TACERA + BL + M6.
Filtered annotation shape: (500, 5)
Unique SampleIds: 500
Unique Patient_ID: 266
Unique Digest: 266


,SampleId,Timepoint,Patient_ID,Digest,Study
0,TAC1241_BL,BL,TAC1241,608CAC12FA322B4798E2E772CE6C9F0087B4933ADB8C3A...,TACERA
1,TAC1241_M6,M6,TAC1241,608CAC12FA322B4798E2E772CE6C9F0087B4933ADB8C3A...,TACERA
2,TAC1147_BL,BL,TAC1147,E856EB9AC7C85B70C9F42C4F6786775FE65DCDFDE2FB71...,TACERA
3,TAC1147_M6,M6,TAC1147,E856EB9AC7C85B70C9F42C4F6786775FE65DCDFDE2FB71...,TACERA
4,TAC1094_BL,BL,TAC1094,10433449E77D5E82E36A52EC5518583272BB0FD55D9BC4...,TACERA


In [22]:
#Cell 5 — sanity check
print(df_annotation_ra_bl["Study"].value_counts(dropna=False))
print(df_annotation_ra_bl["Timepoint"].value_counts(dropna=False))

Study
TACERA    500
Name: count, dtype: int64
Timepoint
BL    265
M6    235
Name: count, dtype: int64


In [23]:
#Cell 6 - selecting relevant sample columns
#Takes fixed metadata columns from the measurement sheet. It keeps: ProteinID, GeneID, Gene Symbol, Gene Name
#Takes the relevant SampleIds from the filtered annotation table and Keeps only those matching sample columns in the measurement sheet
df_measurement_ra_bl = select_relevant_measurement_columns(df_measurement, df_annotation_ra_bl)
df_measurement_ra_bl.head()
print(df_measurement_ra_bl.shape)

Selected relevant measurement columns.
Measurement shape before: (163, 597)
Measurement shape after: (163, 504)
Number of selected sample columns: 500
(163, 504)


In [24]:
#Cell 7 - Create a unique marker name, MarkerName = Gene Symbol + "_" + ProteinID
#Right now your rows are markers, and later after transpose these marker names become the column names.
#We need it so that after the transpose, every future feature column has a clean, unique name
df_measurement_ra_bl = add_marker_name(df_measurement_ra_bl)
df_measurement_ra_bl[["ProteinID", "Gene Symbol", "MarkerName"]].head()

MarkerName column added.
Unique MarkerNames: 163 / 163


,ProteinID,Gene Symbol,MarkerName
0,104740305,XRCC6,XRCC6_104740305
1,104741891,SNRPD1,SNRPD1_104741891
2,104742995,ACTB,ACTB_104742995
3,104743232,PTBP1,PTBP1_104743232
4,104743420,FEN1,FEN1_104743420


In [25]:
#Cell 8 - Transpose the measurement dataset so that rows become columns and columns become rows. This way, every row is a sample, and every column is a marker/feature. This is the format we need for machine learning.
df_measurement_t = transpose_measurement(df_measurement_ra_bl)
df_measurement_t.head()
print(df_measurement_t.shape)

Measurement dataframe transposed.
Shape before transpose: (163, 505)
Shape after transpose: (500, 164)
(500, 164)


In [26]:
#Cell 9 - Mering the transposed measurement table with the filtered annotation table using SampleId.
df_gene_merged = merge_with_annotation(df_measurement_t, df_annotation_ra_bl)
df_gene_merged.head()
print(df_gene_merged.shape)

Merged transposed measurement with annotation.
Transposed measurement shape: (500, 164)
Filtered annotation shape: (500, 5)
Merged shape: (500, 168)
Unique SampleIds after merge: 500
Unique Patient_ID after merge: 266
Unique Digest after merge: 266
(500, 168)


In [27]:
#Cell 10 - Split metadata columns and feature columns since we want to do the cleaning on the feature columns only.
df_meta, df_features = split_metadata_features(df_gene_merged)

df_meta.head()
print(df_meta.shape)
print(df_features.shape)

Split merged dataframe into metadata and features.
Metadata shape: (500, 5)
Feature shape: (500, 163)
(500, 5)
(500, 163)


In [28]:
#Cell 10 - Sanity check on df_meta
print("Metadata shape:", df_meta.shape)
display(df_meta.head())

print("\nUnique SampleIds:", df_meta["SampleId"].nunique())
print("Unique Patient_ID:", df_meta["Patient_ID"].nunique())
print("Unique Digest:", df_meta["Digest"].nunique())

print("\nStudy values:")
print(df_meta["Study"].value_counts(dropna=False))

print("\nTimepoint values:")
print(df_meta["Timepoint"].value_counts(dropna=False))

Metadata shape: (500, 5)


,SampleId,Timepoint,Patient_ID,Digest,Study
0,TAC1241_BL,BL,TAC1241,608CAC12FA322B4798E2E772CE6C9F0087B4933ADB8C3A...,TACERA
1,TAC1241_M6,M6,TAC1241,608CAC12FA322B4798E2E772CE6C9F0087B4933ADB8C3A...,TACERA
2,TAC1147_BL,BL,TAC1147,E856EB9AC7C85B70C9F42C4F6786775FE65DCDFDE2FB71...,TACERA
3,TAC1147_M6,M6,TAC1147,E856EB9AC7C85B70C9F42C4F6786775FE65DCDFDE2FB71...,TACERA
4,TAC1094_BL,BL,TAC1094,10433449E77D5E82E36A52EC5518583272BB0FD55D9BC4...,TACERA



Unique SampleIds: 500
Unique Patient_ID: 266
Unique Digest: 266

Study values:
Study
TACERA    500
Name: count, dtype: int64

Timepoint values:
Timepoint
BL    265
M6    235
Name: count, dtype: int64


In [29]:
#Cell 11 - Sanity check on df_features
print("Feature shape:", df_features.shape)
display(df_features.head())

print("\nDtypes summary:")
print(df_features.dtypes.value_counts())

print("\nBasic stats:")
display(df_features.describe().T.head(10))

Feature shape: (500, 163)


,XRCC6_104740305,SNRPD1_104741891,ACTB_104742995,PTBP1_104743232,FEN1_104743420,EIF4H_104743520,TNC_104745245,IGF1_104745249,FN1_104745436,IGFBP2_104745443,...,APOH_APOH_1113180421,HN1L_HN1L_1043144040,HNRNPA1_HNRNPA1_1066859223,HNRNPA2B1_HNRNPA2B1_1066866713,KDM6B_KDM6B_0172047444,KDM6B_KDM6B_1113180399,MVP_MVP_1066863544,NONO_NONO_0105507292,TMPO_TMPO_1066866329,ZNF574_ZNF574_1066529052
0,116.5,414.0,335.0,254.0,95.0,597.0,289.5,450.0,250.5,712.0,...,168.0,46.0,264.0,171.0,103.0,81.0,2064.0,172.0,134.5,103.0
1,55.0,317.0,146.0,117.0,48.0,244.0,159.0,188.0,160.5,437.0,...,94.0,30.0,115.0,92.0,80.5,51.0,946.0,114.5,78.0,64.0
2,181.5,76.0,236.5,102.5,53.0,130.0,136.0,91.0,167.0,332.0,...,243.0,37.0,103.0,111.0,44.0,61.5,9449.0,276.0,38.0,87.0
3,87.5,45.0,114.0,39.0,22.0,73.0,82.0,48.0,113.0,138.0,...,83.0,19.0,52.0,47.0,21.0,32.0,878.0,116.0,24.0,35.0
4,269.5,424.0,252.5,309.5,166.5,701.5,284.0,537.0,280.0,854.0,...,261.0,97.0,362.0,257.0,117.0,147.0,2128.0,192.0,171.0,140.0



Dtypes summary:
float64    163
Name: count, dtype: int64

Basic stats:


,count,mean,std,min,25%,50%,75%,max
XRCC6_104740305,500.0,192.558,413.937236,18.0,58.875,97.75,171.250,5157.0
SNRPD1_104741891,500.0,236.083,280.982975,26.0,93.375,156.50,273.500,2984.0
ACTB_104742995,500.0,670.281,1861.207456,47.0,151.375,231.50,408.750,16377.0
PTBP1_104743232,500.0,347.219,1252.029634,25.0,94.000,151.75,268.375,22011.0
FEN1_104743420,500.0,119.226,195.956231,12.0,42.000,65.50,115.625,2122.5
EIF4H_104743520,500.0,572.373,1449.795105,48.5,167.000,293.00,541.000,20987.5
TNC_104745245,500.0,262.289,314.889709,41.0,122.000,181.00,290.250,3471.0
IGF1_104745249,500.0,266.686,249.092405,24.0,116.000,185.00,324.000,2147.0
FN1_104745436,500.0,275.942,670.958326,93.0,143.000,174.50,243.500,8986.0
IGFBP2_104745443,500.0,740.099,1168.419397,70.5,227.750,408.00,731.375,10882.0


In [30]:
# Cell 12 - inspect missing values
missing_summary = inspect_missing_values(df_features)

display(missing_summary.head(10))
print("Total missing values in df_features:", df_features.isna().sum().sum())

Missing values inspected.
Feature shape: (500, 163)
Total missing values: 0
Features with any missing values: 0


,feature,missing_count,missing_pct
0,XRCC6_104740305,0,0.0
1,SNRPD1_104741891,0,0.0
2,ACTB_104742995,0,0.0
3,PTBP1_104743232,0,0.0
4,FEN1_104743420,0,0.0
5,EIF4H_104743520,0,0.0
6,TNC_104745245,0,0.0
7,IGF1_104745249,0,0.0
8,FN1_104745436,0,0.0
9,IGFBP2_104745443,0,0.0


Total missing values in df_features: 0


In [31]:
# Cell 13 - inspect low-variance features
variance_summary = inspect_low_variance_features(df_features, threshold=1e-8)

display(variance_summary.head(10))

Low-variance inspection completed.
Feature shape: (500, 163)
Features with variance <= 1e-08: 0


,feature,variance
91,LYZ_1066859023,5280.751378
77,CPSF6_1066836537,6987.248072
98,CTSG_1066859715,7734.407800
32,MBP_1043138774,8435.371241
62,BCAP31_1066528482,9455.498401
68,CXCL5_1066559611,10762.321542
87,FGA_1066858266,12334.298336
70,HIST1H4A_1066561916,13016.153391
73,ZNF217_1066564363,13968.670104
158,KDM6B_KDM6B_1113180399,14140.776524


In [32]:
# Cell 14 - inspect high-correlation features
corr_matrix, high_corr_pairs = inspect_high_correlation_features(df_features, threshold=0.9)

display(high_corr_pairs.head(10))
print("Number of highly correlated pairs:", len(high_corr_pairs))

affected_features = set(high_corr_pairs["feature_1"]).union(set(high_corr_pairs["feature_2"]))
print("Unique affected features:", len(affected_features))
print(sorted(list(affected_features))[:30])

High-correlation inspection completed.
Feature shape: (500, 163)
Highly correlated pairs (>0.9): 52


,feature_1,feature_2,correlation
1370,FN1_104745436,PRTN3_1066558363,0.995315
2230,RPLP2_105481284,RPLP1_1113172457,0.985947
11028,IFNW1_1066559422,BMP7_1086924247,0.985744
23295,CXCL5_1066559611c,HIST1H1B_1066859506c,0.984443
17550,BMP7_1086924247,PLVAP_1086926269,0.982393
2674,CLU_105483601,PRTN3_1066558363,0.980890
2658,CLU_105483601,TUBB_1047887771,0.980776
1320,FN1_104745436,CLU_105483601,0.978170
2690,CLU_105483601,GNPTG_1066840773,0.977684
24110,FGA_1066858266c,HIST1H1B_1066859506c,0.973062


Number of highly correlated pairs: 52
Unique affected features: 29
['ACTB_104742995', 'BCAP31_1066528482', 'BMP7_1086924247', 'CALR_1066859121', 'CENPB_1066861712', 'CLU_105483601', 'CTSG_1066859715', 'CXCL5_1066559611c', 'EHD1_1043170594', 'FGA_1066858266c', 'FN1_104745436', 'GNPTG_1066840773', 'HIST1H1B_1066859506c', 'HIST1H4A_1066561916c', 'HIST2H2AA3_1066858757', 'HIST2H2BE_1066860575', 'IFNW1_1066559422', 'IGFBP6_1066863255', 'LMNB1_1047887413', 'LTF_1066838756', 'LYZ_1066859023', 'MBP_1043138774', 'PLVAP_1086926269', 'PRTN3_1066558363', 'RPLP1_1113172457', 'RPLP2_105481284', 'SNRPN_105510131', 'TNC_104745245', 'TUBB_1047887771']


In [33]:
# Cell 15 - remove high correlation
df_features_hc, dropped_corr_features = remove_high_correlation_features(df_features, threshold=0.9)

print("Shape after high-correlation removal:", df_features_hc.shape)
print("Removed features:", len(dropped_corr_features))
print("First removed features:", dropped_corr_features[:22])

High-correlation removal completed.
Original feature shape: (500, 163)
Reduced feature shape: (500, 143)
Removed highly correlated features: 20
Shape after high-correlation removal: (500, 143)
Removed features: 20
First removed features: ['CLU_105483601', 'SNRPN_105510131', 'LMNB1_1047887413', 'TUBB_1047887771', 'BCAP31_1066528482', 'PRTN3_1066558363', 'LTF_1066838756', 'GNPTG_1066840773', 'HIST2H2AA3_1066858757', 'CALR_1066859121', 'CTSG_1066859715', 'HIST2H2BE_1066860575', 'CENPB_1066861712', 'IGFBP6_1066863255', 'BMP7_1086924247', 'PLVAP_1086926269', 'RPLP1_1113172457', 'HIST1H4A_1066561916c', 'FGA_1066858266c', 'HIST1H1B_1066859506c']


In [34]:
df_features_hc.to_parquet('../../cleaned_datasets/protogen_features.parquet')
df_meta.to_parquet('../../cleaned_datasets/protogen_metadata.parquet')